In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

match_schema = StructType([
    StructField("id", StringType()),
    StructField("utcDate", StringType()),
    StructField("status", StringType()),
    StructField("homeTeam", StructType([
        StructField("name", StringType())
    ])),
    StructField("awayTeam", StructType([
        StructField("name", StringType())
    ])),
    StructField("score", StructType([
        StructField("fullTime", StructType([
            StructField("home", IntegerType()),
            StructField("away", IntegerType())
        ]))
    ])),
    StructField("competition", StructType([
        StructField("name", StringType())
    ]))
])

#Note: to run this notebook is necessary to have Databricks Full access 
#Replace the KAFKA_BOOTSTRAP_SERVERS with your variables 

KAFKA_BOOTSTRAP_SERVERS = "Your KAFKA_BOOTSTRAP_SERVERS"
KAFKA_TOPIC = "football-matches"  #This is specified in the produces.py file
CHECKPOINT_PATH = "/tmp/checkpoints/football_matches_bronze"

df_stream = spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()

df_bronze_stream = df_stream
            .select(from_json(col("value").cast("string"), match_schema).alias("data"))
            .select("data.*")

df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable("football_matches_bronze_stream")
    .start()
    .awaitTermination()




